In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import process_fields
from astropy.visualization import ZScaleInterval

In [ ]:
real_image, simulated_image, real_stars_dao, simulated_stars_dao = (
    process_fields.image_comparison(
        "../../temp/procSPECU2.2019-03-25T05_02_12.640.fits"
    )
)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(25, 15))
interval = ZScaleInterval(contrast=0.05)

vmin, vmax = interval.get_limits(simulated_image)

r_img = axs[0].imshow(real_image, cmap="gray", vmin=vmin, vmax=vmax)
cbar = plt.colorbar(r_img, ax=axs[0], fraction=0.046, pad=0.04)
cbar.set_label("ADU")

s_img = axs[1].imshow(simulated_image, cmap="gray", vmin=vmin, vmax=vmax)
cbar = plt.colorbar(s_img, ax=axs[1], fraction=0.046, pad=0.04)
cbar.set_label("ADU")

d_img = axs[2].imshow(
    real_image.astype(np.float32) - simulated_image.astype(np.float32),
    cmap="gray",
    vmin=-vmax / 5,
    vmax=vmax / 5,
)
cbar = plt.colorbar(d_img, ax=axs[2], fraction=0.046, pad=0.04)
cbar.set_label("ADU")


axs[0].set_title("Real Image")
axs[1].set_title("Simulated Image")
axs[2].set_title("Difference (Real - Simulated)")

fig.savefig("field-comparison.pdf")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 6))

bins = 128
a = np.histogram(real_image.flatten(), bins=bins)
b = np.histogram(simulated_image.flatten(), bins=bins)

axes[0].bar(
    a[1][:-1], a[0], width=np.diff(a[1]), alpha=0.5, label="Real Image", align="edge"
)
axes[0].bar(
    b[1][:-1],
    b[0],
    width=np.diff(b[1]),
    alpha=0.5,
    label="Simulated Image",
    align="edge",
)
axes[0].set_yscale("log")
axes[0].legend()
# axes[0].set_title("Pixel Value Distribution")

axes[1].bar(
    a[1][:-1],
    np.abs(a[0] - b[0]),
    width=np.diff(a[1]),
    label="Difference abs(Real - Simulated)",
    align="edge",
)
axes[1].legend()
axes[1].set_yscale("log")

axes[0].set_xlabel("Pixel Value (ADU)")
axes[0].set_ylabel("Number of Pixels")
axes[1].set_xlabel("Pixel Value (ADU)")
# axes[1].set_ylabel("Number of Pixels")

axes[0].minorticks_on()
axes[1].minorticks_on()

plt.tight_layout()

fig.savefig("field-comparison-histograms.pdf")

In [ ]:
import pandas as pd

df = pd.read_csv("number_of_stars_comparison_threshold-7_20251001_152839.csv")
df["difference"] = df["simulated_stars"] - df["real_stars"]
df["percent_difference"] = (df["difference"] / df["real_stars"]) * 100

print(df["percent_difference"].describe())

plt.figure(figsize=(10, 6))
df["percent_difference"].hist(bins=50, cumulative=False, density=False, log=False)

plt.xlabel("Percent Difference in Number of Stars (Simulated - Real)/Real * 100")
plt.ylabel("Number of Fields")
plt.title("Distribution of Percent Differences in Star Counts")
plt.minorticks_on()

plt.savefig("percent_difference_histogram.pdf")

In [ ]:
df.sort_values("percent_difference")